# Session 1: why a mixture waits, then suddenly ignites
**A. Real-world problem.** Squeeze a fuel-air mixture in an engine and nothing seems to happen
for a short while. Then, within a tiny fraction of that waiting time, the temperature explodes.
That waiting time is the *ignition delay*. We build the smallest model that shows it.

**Important.** This is a one-step *toy*. Real hydrogen-oxygen ignition is a network of
chain-branching reactions with explosion limits; here one made-up reaction stands in for all of it.

**Plan.** (1) a reaction rate that grows steeply with temperature, (2) a closed box where the
reaction heats the gas, (3) step through time and watch the flat-then-jump curve, (4) read off
the ignition delay.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

A_PRE = 2.5e9      # pre-exponential factor, 1/s (toy value)
EA = 150e3         # activation energy, J/mol (toy value)
R_GAS = 8.314      # gas constant, J/(mol K)
DT_AD = 1500.0     # temperature rise if ALL the fuel burns (adiabatic), K
RISE = 400.0       # we call the mixture "ignited" when T has risen by this much, K

## Step 1: a rate that explodes with temperature
The reaction rate constant is `k = A * exp(-Ea / (R * T))` (Arrhenius). Only the few molecules
that collide hard enough react, and the fraction that can grows very fast with T.
Check by hand: at T = 1000 K, `Ea/(R T) = 150000 / (8.314 * 1000) = 18.04`, so
`k = 2.5e9 * exp(-18.04)`, about 36.5 per second. At 1400 K it is about 6300 per second.

In [ ]:
def rate_const(T):
    """Arrhenius rate constant k(T) in 1/s for a temperature T in kelvin."""
    return A_PRE * np.exp(-EA / (R_GAS * T))

## Step 2: the closed box that heats itself
Let `c` be the fuel left (1 = all of it, 0 = none) and `T` the gas temperature.

    dc/dt = -k(T) * c                 fuel is used up at rate k c
    dT/dt = DT_AD * k(T) * c          each bit of fuel burnt heats the gas

No heat escapes (an insulated box). Burning all the fuel would add DT_AD = 1500 K.
*Feedback:* hotter gas gives a bigger k, which burns fuel faster, which heats the gas more.
Take `heat=False` to switch the feedback off and compare.

In [ ]:
def euler_run(T0, t_end, dt, dT_ad=DT_AD, heat=True):
    """Step the box forward with the simplest rule (Euler): new = old + dt * rate.
    Returns arrays t (s), T (K), c (fuel left, 0 to 1)."""
    n = int(round(t_end / dt))
    t = np.arange(n + 1) * dt
    T = np.empty(n + 1)
    c = np.empty(n + 1)
    T[0], c[0] = T0, 1.0
    for i in range(n):
        burn = rate_const(T[i]) * c[i]
        c[i + 1] = c[i] - dt * burn
        T[i + 1] = T[i] + (dt * dT_ad * burn if heat else 0.0)
    return t, T, c

## Step 3: read off the ignition delay
We define the ignition delay `tau` as the time when the temperature has risen by 400 K.
We find it by walking along the curve and interpolating between the two steps that straddle it.

In [ ]:
def ignition_delay(t, T, rise=RISE):
    """Time (s) at which T first reaches T[0] + rise, or None if it never does."""
    hit = np.nonzero(T >= T[0] + rise)[0]
    if len(hit) == 0:
        return None
    j = int(hit[0])
    f = (T[0] + rise - T[j - 1]) / (T[j] - T[j - 1])
    return float(t[j - 1] + f * (t[j] - t[j - 1]))


def euler_delay(T0, dT_ad=DT_AD):
    """Ignition delay (s) from an Euler run that stops at the event. The step is a small
    fraction of the reaction time 1/k at the ignition temperature."""
    if dT_ad <= RISE:          # even burning every drop of fuel cannot raise T by 400 K
        return None
    dt = 0.01 / rate_const(T0 + RISE)
    T, c, t = T0, 1.0, 0.0
    while T < T0 + RISE and t < 100.0:
        burn = rate_const(T) * c
        Tn = T + dt * dT_ad * burn
        if Tn >= T0 + RISE:
            return t + dt * (T0 + RISE - T) / (Tn - T)
        c -= dt * burn
        T = Tn
        t += dt
    return None

## Step 4: see it
Left: temperature against time, with and without the heat feedback. Right: the delay against the
starting temperature (a log scale, because it changes by a factor of more than 100).

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    t, T, c = euler_run(1000.0, 2.0e-3, 0.5e-6)
    t2, T2, c2 = euler_run(1000.0, 2.0e-3, 0.5e-6, heat=False)
    tau = ignition_delay(t, T)
    print(f"T0 = 1000 K: ignition delay = {tau * 1e3:.3f} ms")
    for ms in (0.0, 0.5, 1.0, 1.1, 1.2, 1.3, 1.5):
        i = int(round(ms * 1e-3 / 0.5e-6))
        print(f"  t = {ms:3.1f} ms  T = {T[i]:7.1f} K  fuel left = {c[i]:.3f}")
    print(f"without feedback, T stays at {T2.max():.0f} K and fuel left at 2 ms = {c2[-1]:.3f}")

    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    ax.plot(t * 1e3, T, color="#d9534f", lw=2, label="with heat feedback")
    ax.plot(t2 * 1e3, T2, color="gray", ls="--", label="heat feedback off")
    ax.axvline(tau * 1e3, color="k", ls=":")
    ax.text(tau * 1e3 + 0.03, 1100, f"delay {tau * 1e3:.2f} ms", fontsize=9)
    ax.set_xlabel("time (ms)")
    ax.set_ylabel("temperature (K)")
    ax.set_title("A closed box: flat, then a sudden jump")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "temperature_vs_time.png", dpi=120)

    T0s = np.array([850, 900, 950, 1000, 1050, 1100, 1150, 1200.0])
    taus = np.array([euler_delay(x) for x in T0s])
    for x, y in zip(T0s, taus):
        print(f"  T0 = {x:6.0f} K  delay = {y * 1e3:9.4f} ms")
    fig2, ax2 = plt.subplots(figsize=(6.4, 4.2))
    ax2.semilogy(T0s, taus * 1e3, "o-", color="#1f6feb")
    ax2.set_xlabel("starting temperature T0 (K)")
    ax2.set_ylabel("ignition delay (ms)")
    ax2.set_title("A hotter start ignites much sooner")
    ax2.grid(alpha=0.3, which="both")
    fig2.tight_layout()
    fig2.savefig(OUT / "delay_vs_start.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Run `euler_delay(1000.0, dT)` for heat releases dT = 1000, 1500 and 2000 K and describe the trend in one sentence.
2. **Stretch.** In `euler_delay` change the step from 0.01 / k to 0.2 / k and then to 1 / k. What happens to the answer, and why does a larger step do that?
3. **Explain.** Explain why the box with `heat=False` never ignites, using the two equations of Section B.

## Assessment
- **A. Conceptual.** Why does the mixture wait for a long time and then ignite almost instantly, even though the chemistry is the same all the time? Why does the box without heat feedback never ignite?
- **B. Numerical.** Find the rate constant at 1100 K. By what factor is it bigger than at 1000 K? If 10 percent of the fuel burns, by how many kelvin does the box heat up?
- **C. Graph and simulation interpretation.** In the right-hand figure, by what factor does the delay shrink when the start temperature rises from 1000 K to 1050 K? What does the steepness of the curve tell you about the rate?
- **D. Design and optimisation challenge.** Design: you need ignition within 0.5 ms. Find the lowest start temperature (in steps of 10 K) that works with a heat release of 1500 K. Then find out whether a richer mixture with 2000 K heat release lets you start cooler, and say what you give up.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def _reference_delay(T0, dT_ad=1500.0, rise=400.0, A=2.5e9, Ea=150e3, R=8.314):
    """Independent reference: because T + dT_ad*c is constant, tau = integral of dc / (k c)."""
    x, w = np.polynomial.legendre.leggauss(100)
    cs = 1 - rise / dT_ad
    c = 0.5 * (1 - cs) * x + 0.5 * (1 + cs)
    k = A * np.exp(-Ea / (R * (T0 + dT_ad * (1 - c))))
    return 0.5 * (1 - cs) * float(np.sum(w / (k * c)))


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("hand calculation: Ea/(R T) = 18.04 and k(1000 K) = 36.5 per second",
                   abs(L.EA / (L.R_GAS * 1000.0) - 18.04) < 0.005 and abs(L.rate_const(1000.0) - 36.5) < 0.1))
    checks.append(("k(1400 K) is about 6300 per second, 170 times k(1000 K)",
                   abs(L.rate_const(1400.0) - 6330) < 30 and 170 < L.rate_const(1400.0) / L.rate_const(1000.0) < 175))
    t0, T0, c0 = L.euler_run(1000.0, 1.0e-3, 0.5e-6, heat=False)
    checks.append(("worked example: without heat feedback only 3.6 percent of the fuel is gone after 1 ms",
                   abs(c0[-1] - 0.964) < 0.001 and abs(T0[-1] - 1000.0) < 1e-9))
    t, T, c = L.euler_run(1000.0, 2.0e-3, 0.5e-6)
    tau = L.ignition_delay(t, T)
    checks.append(("worked example: with feedback the delay at 1000 K is 1.19 ms",
                   abs(tau * 1e3 - 1.19) < 0.02))
    checks.append(("Euler delay agrees with the independent reference within 1 percent",
                   abs(tau / _reference_delay(1000.0) - 1) < 0.01))
    checks.append(("energy bookkeeping: T + 1500 c stays equal to T0 + 1500",
                   np.allclose(T + L.DT_AD * c, 1000.0 + L.DT_AD, atol=1e-6)))
    i = int(round(1.0e-3 / 0.5e-6))
    j = int(round(1.3e-3 / 0.5e-6))
    checks.append(("flat then jump: T is below 1140 K at 1.0 ms and 2500 K at 1.3 ms",
                   T[i] < 1140 and abs(T[j] - 2500.0) < 1.0))
    checks.append(("at ignition 26.7 percent of the fuel is burnt (400/1500)",
                   abs((1 - c[np.argmax(T >= 1400.0)]) - 0.267) < 0.01))
    d = [L.euler_delay(x) for x in (850.0, 1000.0, 1150.0)]
    checks.append(("a hotter start ignites sooner, and 850 K waits over 100 times longer than 1150 K",
                   d[0] > d[1] > d[2] and d[0] / d[2] > 100))
    checks.append(("edge: if burning everything gives only 300 K, there is no ignition",
                   L.euler_delay(1000.0, 300.0) is None))
    checks.append(("k(1100 K) = 188 per second, 5.2 times k(1000 K); burning 10 percent of the fuel adds 150 K",
                   abs(L.rate_const(1100.0) - 188.27) < 0.1 and abs(L.rate_const(1100.0) / L.rate_const(1000.0) - 5.156) < 0.01
                   and abs(0.1 * L.DT_AD - 150.0) < 1e-9))
    checks.append(("the curve: T is 1134 K at 1.0 ms and 1458 K at 1.2 ms",
                   abs(T[i] - 1133.7) < 1.0 and abs(T[int(round(1.2e-3 / 0.5e-6))] - 1458.5) < 2.0))
    checks.append(("delays: 900 K gives 7.06 ms and 1100 K gives 0.284 ms; 1000 K to 1050 K cuts the delay by a factor of 2.1",
                   abs(L.euler_delay(900.0) * 1e3 - 7.062) < 0.02 and abs(L.euler_delay(1100.0) * 1e3 - 0.2843) < 0.002
                   and abs(L.euler_delay(1000.0) / L.euler_delay(1050.0) - 2.126) < 0.01))
    checks.append(("design: 1050 K is too slow for 0.5 ms (0.562 ms), 1060 K passes (0.488 ms); a 2000 K rise passes at 1050 K (0.416 ms)",
                   L.euler_delay(1050.0) > 0.5e-3 > L.euler_delay(1060.0) and abs(L.euler_delay(1060.0) * 1e3 - 0.488) < 0.002
                   and abs(L.euler_delay(1050.0, 2000.0) * 1e3 - 0.416) < 0.002))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
